# Abstention Machine Translation Scoring
Generates machine translation metric scores (e.g., BLEU) for sampled captions. We think these scores can be appropriate for this setting since we're looking at model self-similarity as a metric for whether they should abstain

In [1]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("../")

import os
import copy
import json
from datetime import datetime
import time
from tqdm.notebook import tqdm

import torch

from evaluation.evaluate_captions import (
    execute_bleu,
    execute_meteor,
    execute_rouge,
    execute_cider,
    execute_spice,
    execute_bertscore,
)

2025-08-12 12:09:18.952627: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755025759.382609 1665916 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755025759.475607 1665916 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755025760.400004 1665916 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755025760.400031 1665916 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755025760.400036 1665916 computation_placer.cc:177] computation placer alr

Using device: cuda:0


In [2]:
# setup pytorch for BERTScore
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # for multi-GPU systems, force single GPU
if torch.cuda.is_available():
    device_type = "cuda:0"  # force single, first GPU
elif torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"
print(f"Using device: {device_type}")

Using device: cuda:0


In [3]:
# load data
with open("./results/test-data-scored_300-examples_2025-07-29_14-40-05.json", "r") as f:
    input_data_dict = json.load(f)
input_data_dict[0]

{'image_id': 1308,
 'file_name': 'VizWiz_train_00001308.jpg',
 'vizwiz_url': 'https://vizwiz.cs.colorado.edu/VizWiz_visualization_img/VizWiz_train_00001308.jpg',
 'human_captions': 'a small 330 ml bottle of praise French salad dressing\nA picture of the food is on the packaging.\nSmall blue and green French dressing bottle laying sideways.\nA person holding a bottle of French dressing by Praise in green and blue lettering on a tan placemat.\na bottle of French salad dressing brand by praise',
 'annotator': 'Anne Marie',
 'annotation': 'praise french dressing',
 'gpt4o_caption': 'A bottle of Praise French dressing with a green cap, featuring a blue label with text "French" and product details. The bottle is being held over a woven mat.',
 'gpt4o_code': 'yes',
 'greedy_response': 'A bottle of Praise French dressing with a green cap and a blue label. The label includes images of garlic and herbs, and text indicating it is a 330ml bottle with no preservatives, artificial colors, or flavors

In [16]:
# remove stopwords before processing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
stopwords = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /home/kapilg/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
def process_tokens(sentence, stopwords):
    # get tokens
    tokens = word_tokenize(sentence)

    # strip punctuation from tokens
    words = [word.lower() for word in tokens if word.isalpha()]

    # remove stop words
    return [word for word in words if word not in stopwords]

In [27]:
# count the words in the caption
for index, image in enumerate(input_data_dict):
    image["greedy_response_no_stop"] = " ".join(
        process_tokens(image["greedy_response"], stopwords)
    )

    image["additional_responses_no_stop"] = []
    for add_resp in image["additional_responses"]:
        image["additional_responses_no_stop"].append(
            " ".join(process_tokens(add_resp, stopwords))
        )

In [30]:
# create a long list of candidates and references that can be passed to each scoring function
candidates = []
references = []
indices = []  # used for merging results into
for index, image_result in enumerate(tqdm(input_data_dict)):
    # candidate sentences are the generated samples
    # references are a list of list of str, where each inner list is just the greedy response
    curr_candidates = image_result["additional_responses_no_stop"]
    curr_references = [[image_result["greedy_response_no_stop"]]] * len(curr_candidates)

    candidates += curr_candidates
    references += curr_references
    indices.append(len(curr_candidates))

  0%|          | 0/300 [00:00<?, ?it/s]

In [31]:
output_scores = {}

In [32]:
start_time = time.time()
output_scores["bleu-1"] = execute_bleu(candidates, references, 1)
output_scores["bleu-2"] = execute_bleu(candidates, references, 2)
output_scores["bleu-3"] = execute_bleu(candidates, references, 3)
output_scores["bleu-4"] = execute_bleu(candidates, references, 4)
print(
    f"--- BLEU 1-4: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

--- BLEU 1-4: 160.36 seconds for 300 images (2995 total samples) ---


In [33]:
start_time = time.time()
scores = execute_meteor(candidates, references)
output_scores["meteor"] = [{"score": float(x["meteor"])} for x in scores]
print(
    f"--- Meteor: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

[nltk_data] Downloading package wordnet to /home/kapilg/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/kapilg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/kapilg/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


--- Meteor: 44.25 seconds for 300 images (2995 total samples) ---


In [34]:
start_time = time.time()
scores = execute_rouge(candidates, references)
output_scores["rouge"] = [
    {
        "rouge1": float(x["rouge1"]),
        "rouge2": float(x["rouge2"]),
        "rougeL": float(x["rougeL"]),
        "rougeLsum": float(x["rougeLsum"]),
    }
    for x in scores
]
print(
    f"--- Rogue: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

--- Rogue: 285.42 seconds for 300 images (2995 total samples) ---


In [35]:
start_time = time.time()
_, scores = execute_cider(candidates, references)
output_scores["cider"] = [{"score": float(x)} for x in scores]
print(
    f"--- CIDEr: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

PTBTokenizer tokenized 52234 tokens at 359583.58 tokens per second.
PTBTokenizer tokenized 54724 tokens at 449661.28 tokens per second.


--- CIDEr: 1.35 seconds for 300 images (2995 total samples) ---


In [36]:
start_time = time.time()
_, scores = execute_spice(candidates, references)
output_scores["spice"] = scores
print(
    f"--- SPICE: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

PTBTokenizer tokenized 52234 tokens at 450450.71 tokens per second.
PTBTokenizer tokenized 54724 tokens at 449260.67 tokens per second.
Parsing reference captions
Initiating Stanford parsing pipeline
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator tokenize
[main] INFO edu.stanford.nlp.pipeline.TokenizerAnnotator - TokenizerAnnotator: No tokenizer type provided. Defaulting to PTBTokenizer.
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ssplit
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator parse
[main] INFO edu.stanford.nlp.parser.common.ParserGrammar - Loading parser from serialized file edu/stanford/nlp/models/lexparser/englishPCFG.ser.gz ... 
done [0.4 sec].
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator lemma
[main] INFO edu.stanford.nlp.pipeline.StanfordCoreNLP - Adding annotator ner
Loading classifier from edu/stanford/nlp/models/ner/english.all.3class.distsim.crf.ser.gz ... done 

SPICE evaluation took: 1.362 min
--- SPICE: 83.43 seconds for 300 images (2995 total samples) ---


In [37]:
start_time = time.time()
output_scores["bertscore"] = execute_bertscore(
    candidates, references, device=device_type
)
print(
    f"--- BERTScore: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

--- BERTScore: 38.96 seconds for 300 images (2995 total samples) ---


In [38]:
start_time = time.time()
output_scores["bertscore_idf"] = execute_bertscore(
    candidates, references, device=device_type, idf=True
)
print(
    f"--- BERTScore with IDF weighting: {(time.time() - start_time):.2f} seconds for {len(input_data_dict)} images ({len(candidates)} total samples) ---"
)

--- BERTScore with IDF weighting: 9.64 seconds for 300 images (2995 total samples) ---


In [39]:
# combine
output_start_ptr = 0
output_end_ptr = output_start_ptr
output_data_dict = copy.deepcopy(input_data_dict)

metrics = [
    "bleu-1",
    "bleu-2",
    "bleu-3",
    "bleu-4",
    "meteor",
    "rouge",
    "cider",
    "spice",
    "bertscore",
    "bertscore_idf",
]

for input_index in range(len(output_data_dict)):
    curr_index = indices[input_index]
    output_end_ptr = output_start_ptr + curr_index

    for metric in metrics:
        output_data_dict[input_index][metric] = output_scores[metric][
            output_start_ptr:output_end_ptr
        ]
        assert len(output_data_dict[input_index][metric]) == len(
            output_data_dict[input_index]["additional_responses_no_stop"]
        )

    # increment the start pointer
    output_start_ptr = output_end_ptr

In [40]:
# save as json
os.makedirs("results", exist_ok=True)
with open(
    f"results/test-data-scored-with-mt-metrics_{len(output_data_dict)}-examples_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.json",
    "w",
) as f:
    json.dump(output_data_dict, f, indent=2)